****AKLT investigation with my own MPS instance****

First we define the AKLT state

In [ ]:
from mpsdmrg import MPS, SpinOperator
import torch
from math import sqrt
import numpy as np

sp = torch.tensor([[0.,1.],[0.,0.]], dtype=torch.complex128,device='cuda')
sm = torch.tensor([[0.,0.],[1.,0.]], dtype=torch.complex128,device='cuda')
sz = torch.tensor([[1.,0.],[0.,-1.]], dtype=torch.complex128,device='cuda')

A = torch.zeros(2, 3, 2, dtype=torch.complex128,device='cuda')
A[:,0,:] = sp
A[:,1,:] = -sz / sqrt(2)
A[:,2,:] = -sm

L = 100
tensors = [A.clone() for _ in range(L)]
bond_dims = [2]*(L+1)
aklt = MPS(L, 3, bond_dims, init_state=tensors, dtype=torch.complex128)

First let us see that there is no magnetization.

In [ ]:
m = np.zeros((3, L)) 
S = SpinOperator(3).data # spin 1 operators Sx, Sy, Sz

for i in range(L):
    for j in range(3):
        m[j, i] = aklt.expectation({i: S[j]}).real
print("Magnetization:", np.max(np.abs(m)))

Next we measure the correlation

In [ ]:

zzcorr = np.zeros(L)
for i in range(L):
    res = aklt.expectation({i: S[2], 50: S[2]})
    # convert complex result to real float
    if isinstance(res, torch.Tensor):
        val = res.cpu().real.item()
    else:
        val = float(np.real(res))
    zzcorr[i] = val


In [ ]:
import matplotlib.pyplot as plt
plt.plot(range(L//2-1), abs(zzcorr)[L//2+1:])
plt.yscale('log')

from scipy.stats import linregress
x = np.arange(1, L//2)
y = np.log(abs(zzcorr)[L//2+1:])
slope, intercept, r_value, p_value, std_err = linregress(x, y)
print("Slope:", slope, 'compared with ln(1/3) =', np.log(1/3))


Finally String order:

In [ ]:
Ostr = torch.diag(torch.tensor([-1,1,-1], dtype=torch.complex128,device='cuda'))

for j in range(1,L):
    string_order = {
    0: S[2]
}
    for i in range(1,j):
        string_order[i] = Ostr.clone()
    string_order[j] = S[2]
    print(f"String order parameter <Sz(0)O^{j-1}Sz({j})> = {aklt.expectation(string_order)}")
    # converges to -4/9

## Sweeping the AKLT Hamiltonian

The valence-bond state above is the exact ground state of the bilinear-biquadratic Hamiltonian $H = \sum_i (\vec S_i\cdot\vec S_{i+1} + \frac{1}{3}(\vec S_i\cdot\vec S_{i+1})^2)$, each bond contributing $-2/3$, so an open chain of $L$ sites has $E = -\frac{2}{3}(L-1)$.  
Sweeping that Hamiltonian with SU(2) symmetry, from a random singlet, must recover the same energy.

In [ ]:
from mpsdmrg import Broomstick
from mpsdmrg.symmetry import su2_symmetry

La = 6
aklt_stick = Broomstick(L=La, phys_dim=3, max_bond_dim=8, svd_tol=1e-14, group=su2_symmetry)
for i in range(La - 1):
    for a in 'XYZ':
        aklt_stick.add_couplings((a, a), 1.0, (i, i + 1))
    for a in 'XYZ':
        for b in 'XYZ':
            aklt_stick.add_couplings((a + b, a + b), 1.0 / 3.0, (i, i + 1))
aklt_stick.build()
aklt_stick.sweep(num_sweeps=4)
print(f'AKLT  DMRG {aklt_stick.energy:.12f}  exact {-2 / 3 * (La - 1):.12f}')

## Sweeping the ULS chain

The Uimin-Lai-Sutherland chain is the SU(3) symmetric spin-1 chain $H = \sum_i P_{i,i+1}$, where $P$ permutes the two sites, $P|a,b\rangle = |b,a\rangle$.  
In spin operators that permutation is $P = -1 + \vec S_i\cdot\vec S_{i+1} + (\vec S_i\cdot\vec S_{i+1})^2$, so the same builder that sweeps AKLT sweeps the critical ULS chain, whose energy density approaches $\frac{1}{4} - \ln 3$.

In [ ]:
print('  L       DMRG    per bond')
for Lu in (4, 6):
    uls = Broomstick(L=Lu, phys_dim=3, max_bond_dim=16, svd_tol=1e-14, group=su2_symmetry)
    for i in range(Lu - 1):
        uls.add_couplings(('I', 'I'), -1.0, (i, i + 1))
        for a in 'XYZ':
            uls.add_couplings((a, a), 1.0, (i, i + 1))
        for a in 'XYZ':
            for b in 'XYZ':
                uls.add_couplings((a + b, a + b), 1.0, (i, i + 1))
    uls.build()
    uls.sweep(num_sweeps=4)
    print(f'{Lu:3d}  {uls.energy:10.6f}  {uls.energy / (Lu - 1):8.6f} / bond')
print('thermodynamic limit 1/4 - ln 3 =', 0.25 - np.log(3))